# RAW HDR Highlight Recovery — Fingerprint Training

**Strategy:** Synthetic Distillation on 64×64 Bayer fingerprint patches

- Upload your RAW files (.dng / .nef / .cr2 / .arw) to `/content/fivek_raw/` via the Files panel on the left.
- Output: `raw_hdr_recovery.bin` — weight file for the StudioRoom C++ RAZ1 inference engine.

**Before running:** Runtime → Change runtime type → **T4 GPU**

**Run all cells top to bottom. Do not skip any cell.**

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q rawpy tifffile imageio
print('Dependencies installed.')

In [ ]:
# ── Cell 2: Prepare RAW directory ─────────────────────────────────────────────
# Upload your .dng / .nef / .cr2 / .arw files to /content/fivek_raw/ using
# the Files panel on the left side of Colab, then run this cell.

from pathlib import Path

RAW_DIR = Path('/content/fivek_raw')
RAW_DIR.mkdir(exist_ok=True)

raw_files = []
for ext in ('*.dng','*.DNG','*.nef','*.NEF','*.cr2','*.CR2','*.arw','*.ARW'):
    raw_files.extend(sorted(RAW_DIR.glob(ext)))

print(f'Found {len(raw_files)} RAW files in {RAW_DIR}:')
for f in raw_files:
    print(f'  {f.name}  ({f.stat().st_size/1e6:.1f} MB)')

if len(raw_files) == 0:
    print()
    print('WARNING: No RAW files found!')
    print('Upload .dng/.nef/.cr2/.arw files to /content/fivek_raw/ and re-run this cell.')

In [ ]:
# ── Cell 3: Extract 64×64 fingerprint patches ─────────────────────────────────
# Only patches with ≥10% clipped pixels are kept (highlight fingerprints).
# Stride = 32 (50% overlap) → ~4× more patches than non-overlapping.

import rawpy
import numpy as np
from pathlib import Path

PATCH_SIZE  = 64    # must match C++ tile size
STRIDE      = 32    # 50% overlap
CLIP_FRAC   = 0.92  # pixels above this are considered clipped
DENSITY_MIN = 0.10  # minimum fraction of clipped pixels per patch
MAX_PATCHES = 200   # max patches per RAW file

PATCH_DIR = Path('/content/fingerprints')
PATCH_DIR.mkdir(exist_ok=True)

# Clear old patches
for f in PATCH_DIR.glob('*.npy'): f.unlink()

raw_files = []
for ext in ('*.dng','*.DNG','*.nef','*.NEF','*.cr2','*.CR2','*.arw','*.ARW'):
    raw_files.extend(sorted(Path('/content/fivek_raw').glob(ext)))
print(f'Processing {len(raw_files)} RAW files ...')

total = 0
for rp in raw_files:
    try:
        with rawpy.imread(str(rp)) as raw:
            bayer = raw.raw_image.copy()
            white = raw.white_level
            black = int(np.median(raw.black_level_per_channel))
    except Exception as e:
        print(f'  SKIP {rp.name}: {e}'); continue

    norm = np.clip((bayer.astype(np.float32) - black) / max(white - black, 1), 0, 1)
    H, W = norm.shape
    saved = 0
    for y in range(0, H - PATCH_SIZE + 1, STRIDE):
        for x in range(0, W - PATCH_SIZE + 1, STRIDE):
            if saved >= MAX_PATCHES: break
            patch = norm[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
            if float((patch > CLIP_FRAC).mean()) < DENSITY_MIN: continue
            np.save(str(PATCH_DIR / f'{rp.stem}_y{y:04d}_x{x:04d}.npy'),
                    patch.astype(np.float32))
            saved += 1; total += 1
        if saved >= MAX_PATCHES: break
    if saved: print(f'  {rp.name}: {saved} patches')

print(f'\nTotal fingerprint patches: {total}')
print(f'Dataset size: {total * PATCH_SIZE * PATCH_SIZE * 4 / 1e6:.1f} MB')
if total < 50:
    print('WARNING: very few patches — add more RAW files or lower DENSITY_MIN')

In [ ]:
# ── Cell 4: Dataset and DataLoaders ───────────────────────────────────────────
# Each patch: [64,64] Bayer → pixel-unshuffle → [4,32,32] RGGB.
# Synthetic clip at random level 0.60–0.92 creates (clipped_input, gt) pairs.
# Dihedral augmentation (flips + 90° rotations) applied only during training.
# Safe on pixel-unshuffled data: channels are already separated, spatial
# transforms on dims [1,2] preserve every channel's RGGB identity.

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
from pathlib import Path

class FingerprintDataset(Dataset):
    CLIP_LO = 0.60
    CLIP_HI = 0.92

    def __init__(self, paths, augment=True):
        self.paths   = paths
        self.augment = augment

    def __len__(self): return len(self.paths)

    @staticmethod
    def _unshuffle(p):
        return np.stack([p[0::2,0::2], p[0::2,1::2],
                         p[1::2,0::2], p[1::2,1::2]], axis=0).astype(np.float32)

    @staticmethod
    def _dihedral(t):
        if random.random() > 0.5: t = torch.flip(t, dims=[2])
        if random.random() > 0.5: t = torch.flip(t, dims=[1])
        k = random.randint(0, 3)
        if k: t = torch.rot90(t, k=k, dims=[1, 2])
        return t

    def __getitem__(self, idx):
        gt = torch.from_numpy(self._unshuffle(np.load(str(self.paths[idx]))))
        if self.augment: gt = self._dihedral(gt)
        lvl = random.uniform(self.CLIP_LO, self.CLIP_HI)
        return torch.clamp(gt, max=lvl), gt


all_patches = sorted(Path('/content/fingerprints').glob('*.npy'))
random.shuffle(all_patches)
n_val = max(20, len(all_patches) // 10)
val_p, trn_p = all_patches[:n_val], all_patches[n_val:]

ds_trn = FingerprintDataset(trn_p, augment=True)
ds_val = FingerprintDataset(val_p, augment=False)
dl_trn = DataLoader(ds_trn, batch_size=64, shuffle=True,  num_workers=2, pin_memory=True)
dl_val = DataLoader(ds_val, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train patches : {len(trn_p)}')
print(f'Val patches   : {len(val_p)}')
print(f'Batches/epoch : {len(dl_trn)}')
print(f'Effective samples/epoch (D4 aug): ~{len(trn_p)*8:,}')

In [ ]:
# ── Cell 5: Model definition ───────────────────────────────────────────────────
# Depthwise-separable U-Net, <15K params, mirrors raw_hdr_recovery.cpp exactly.
# Input:  [B, 4, 32, 32]  pixel-unshuffled RGGB
# Output: [B, 4, 32, 32]  recovered RGGB, sigmoid [0,1]

import torch
import torch.nn as nn
import torch.nn.functional as F

class DWSepBlock(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.dw = nn.Conv2d(c_in, c_in,  3, padding=1, groups=c_in, bias=True)
        self.pw = nn.Conv2d(c_in, c_out, 1,                         bias=True)
        self.bn = nn.BatchNorm2d(c_out)
    def forward(self, x):
        return F.relu(self.bn(self.pw(self.dw(x))), inplace=True)

class RawHdrUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc0 = DWSepBlock(4,       16)
        self.enc1 = DWSepBlock(16,      32)
        self.enc2 = DWSepBlock(32,      64)
        self.bot  = DWSepBlock(64,      64)
        self.dec2 = DWSepBlock(64 + 32, 32)
        self.dec1 = DWSepBlock(32 + 16, 16)
        self.head = nn.Conv2d(16, 4, 1)

    def forward(self, x):
        e0 = self.enc0(x)
        e1 = self.enc1(F.max_pool2d(e0, 2))
        e2 = self.enc2(F.max_pool2d(e1, 2))
        b  = self.bot(e2)
        d2 = self.dec2(torch.cat([F.interpolate(b,  scale_factor=2, mode='bilinear', align_corners=False), e1], 1))
        d1 = self.dec1(torch.cat([F.interpolate(d2, scale_factor=2, mode='bilinear', align_corners=False), e0], 1))
        return torch.sigmoid(self.head(d1))

model = RawHdrUNet()
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')
assert n_params == 13_900, f'Architecture mismatch: got {n_params}'

In [ ]:
# ── Cell 6: Training ───────────────────────────────────────────────────────────
# Loss = highlight-weighted L1 (8× clipped, 1× unclipped) + 0.1 × VGG perceptual
# Optimizer: AdamW + OneCycleLR over 200 epochs.
# Best checkpoint saved to /content/raw_hdr_unet.pth

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features
        self.slice = nn.Sequential(*list(vgg.children())[:16])
        for p in self.slice.parameters(): p.requires_grad = False
        self.slice.eval()
        self.register_buffer('mean', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))

    def forward(self, rec, tgt):
        def to_rgb(x):
            return torch.cat([x[:,0:1], 0.5*x[:,1:2]+0.5*x[:,2:3], x[:,3:4]], 1)
        def prep(x):
            x = F.interpolate(to_rgb(x), size=(64,64), mode='bilinear', align_corners=False)
            return (x - self.mean) / self.std
        return F.l1_loss(self.slice(prep(rec)), self.slice(prep(tgt)))


device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}', torch.cuda.get_device_name(0) if device.type=='cuda' else '')

EPOCHS     = 200
CLIP_LEVEL = 0.60
LAMBDA_VGG = 0.1
CKPT       = '/content/raw_hdr_unet.pth'

model    = model.to(device)
vgg_loss = VGGPerceptualLoss().to(device)
opt      = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)
sched    = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=3e-3, steps_per_epoch=len(dl_trn), epochs=EPOCHS,
    pct_start=0.05, div_factor=25, final_div_factor=1000)

best_val = float('inf')
history  = {'train': [], 'val': []}

for epoch in range(1, EPOCHS + 1):
    model.train(); vgg_loss.eval()
    t_loss = 0.
    for inp, gt in dl_trn:
        inp, gt = inp.to(device), gt.to(device)
        pred   = model(inp)
        mask   = (inp >= CLIP_LEVEL).float()
        l_pix  = 8.*F.l1_loss(pred*mask, gt*mask) + 1.*F.l1_loss(pred*(1-mask), gt*(1-mask))
        loss   = l_pix + LAMBDA_VGG * vgg_loss(pred, gt)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        opt.step(); sched.step()
        t_loss += loss.item()

    model.eval()
    v_loss = 0.
    with torch.no_grad():
        for inp, gt in dl_val:
            inp, gt = inp.to(device), gt.to(device)
            pred  = model(inp)
            mask  = (inp >= CLIP_LEVEL).float()
            l_pix = 8.*F.l1_loss(pred*mask, gt*mask) + 1.*F.l1_loss(pred*(1-mask), gt*(1-mask))
            v_loss += (l_pix + LAMBDA_VGG * vgg_loss(pred, gt)).item()

    tl = t_loss / len(dl_trn)
    vl = v_loss / len(dl_val)
    history['train'].append(tl)
    history['val'].append(vl)
    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), CKPT)
    if epoch % 25 == 0 or epoch <= 3:
        print(f'Ep {epoch:3d}/{EPOCHS}  train={tl:.5f}  val={vl:.5f}  lr={sched.get_last_lr()[0]:.2e}  best={best_val:.5f}')

print(f'\nDone. Best val loss: {best_val:.5f}')

In [ ]:
# ── Cell 7: Loss curve ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 4))
plt.plot(history['train'], label='train')
plt.plot(history['val'],   label='val')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(True)
plt.title('RAW HDR Recovery — Training Curve')
plt.tight_layout()
plt.savefig('/content/training_curve.png', dpi=100)
plt.show()

In [ ]:
# ── Cell 8: Visual quality check ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

model.load_state_dict(torch.load(CKPT, map_location=device))
model.eval()
inp_b, gt_b = next(iter(dl_val))
with torch.no_grad():
    pred_b = model(inp_b.to(device)).cpu()

def thumb(t):
    r, gr, gb, b = t[0].numpy(), t[1].numpy(), t[2].numpy(), t[3].numpy()
    return np.clip(np.stack([r, (gr+gb)/2, b], axis=-1), 0, 1)

fig, axes = plt.subplots(4, 3, figsize=(9, 12))
for i in range(4):
    axes[i,0].imshow(thumb(inp_b[i]));  axes[i,0].set_title('Input (clipped)',  fontsize=9)
    axes[i,1].imshow(thumb(pred_b[i])); axes[i,1].set_title('Recovered',        fontsize=9)
    axes[i,2].imshow(thumb(gt_b[i]));   axes[i,2].set_title('Ground truth',     fontsize=9)
    for ax in axes[i]: ax.axis('off')
plt.suptitle('Highlight recovery — validation set', y=1.01)
plt.tight_layout()
plt.savefig('/content/quality_check.png', dpi=100)
plt.show()

In [ ]:
# ── Cell 9: Export to RAZ1 binary ─────────────────────────────────────────────
# Fuses BatchNorm into affine scale+bias. Output consumed by raw_hdr_recovery.cpp.

import struct, io
import numpy as np
from pathlib import Path

model.load_state_dict(torch.load(CKPT, map_location='cpu'))
model.eval()
sd = model.state_dict()

tensors, shapes = {}, {}

def fuse_bn(prefix):
    dw_w  = sd[f'{prefix}.dw.weight']
    dw_b  = sd[f'{prefix}.dw.bias']
    pw_w  = sd[f'{prefix}.pw.weight']
    pw_b  = sd[f'{prefix}.pw.bias']
    C, k  = dw_w.shape[0], dw_w.shape[2]
    C_out = pw_w.shape[0]
    gamma = sd[f'{prefix}.bn.weight'].numpy()
    beta  = sd[f'{prefix}.bn.bias'].numpy()
    mu    = sd[f'{prefix}.bn.running_mean'].numpy()
    var   = sd[f'{prefix}.bn.running_var'].numpy()
    scale = gamma / np.sqrt(var + 1e-5)
    bias  = beta - mu * scale
    tensors[f'{prefix}.dw.weight'] = dw_w.reshape(C, k, k).numpy();  shapes[f'{prefix}.dw.weight'] = [C, k, k]
    tensors[f'{prefix}.dw.bias']   = dw_b.numpy();                    shapes[f'{prefix}.dw.bias']   = [C]
    tensors[f'{prefix}.pw.weight'] = pw_w.reshape(C_out, C).numpy();  shapes[f'{prefix}.pw.weight'] = [C_out, C]
    tensors[f'{prefix}.pw.bias']   = pw_b.numpy();                    shapes[f'{prefix}.pw.bias']   = [C_out]
    tensors[f'{prefix}.bn.scale']  = scale;                           shapes[f'{prefix}.bn.scale']  = [C_out]
    tensors[f'{prefix}.bn.bias']   = bias;                            shapes[f'{prefix}.bn.bias']   = [C_out]

for blk in ['enc0','enc1','enc2','bot','dec2','dec1']:
    fuse_bn(blk)

tensors['head.weight'] = sd['head.weight'].reshape(4, 16).numpy(); shapes['head.weight'] = [4, 16]
tensors['head.bias']   = sd['head.bias'].numpy();                  shapes['head.bias']   = [4]

buf = io.BytesIO()
buf.write(b'RAZ1')
buf.write(struct.pack('<II', 1, len(tensors)))
for name, data in tensors.items():
    nb = name.encode('utf-8')
    sh = shapes[name]
    fl = data.astype(np.float32).flatten()
    buf.write(struct.pack('<I', len(nb))); buf.write(nb)
    buf.write(struct.pack('<I', len(sh)))
    for d in sh: buf.write(struct.pack('<I', d))
    buf.write(fl.tobytes())

Path('/content/raw_hdr_recovery.bin').write_bytes(buf.getvalue())
total_params = sum(int(np.prod(shapes[k])) for k in shapes)
print(f'Tensors  : {len(tensors)}')
print(f'Params   : {total_params:,}')
print(f'File size: {len(buf.getvalue())/1024:.1f} KB')

In [ ]:
# ── Cell 10: Download ─────────────────────────────────────────────────────────
from google.colab import files
files.download('/content/raw_hdr_recovery.bin')
files.download('/content/training_curve.png')
files.download('/content/quality_check.png')
print('Place raw_hdr_recovery.bin in:')
print('  feature/photo-editor/src/main/assets/models/raw_hdr_recovery.bin')